In [0]:
%sql
DESCRIBE jarvis_etl.silver.transactions;

col_name,data_type,comment
id,int,null
client_id,int,null
card_id,int,null
amount,"decimal(10,2)",null
use_chip,string,null
merchant_id,int,null
merchant_city,string,null
merchant_state,string,null
zip,double,null
mcc,int,null


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS jarvis_etl.gold;

## Q1: Which day of week has highest fraudulent transactions

In [0]:
from pyspark.sql.functions import dayofweek, date_format, col

transactions = spark.table("jarvis_etl.silver.transactions")

fraud_by_dow = (
    transactions
    .filter(col("is_fraud") == "Yes")
    .groupBy(dayofweek(col("transaction_date")).alias("day_number"), date_format(col("transaction_date"), "EEEE").alias("day_of_week"))
    .count()
    .orderBy("day_number")
)

fraud_by_dow.write.mode("overwrite").saveAsTable("jarvis_etl.gold.fraud_by_day_of_week")

## Q2: Fraud rate trend over time by month

In [0]:
from pyspark.sql.functions import date_format, avg, when

fraud_trend = (
    transactions
    .groupBy(date_format(col("transaction_date"), "yyyy-MM").alias("month"))
    .agg(
        avg(when(col("is_fraud") == "Yes", 1).otherwise(0)).alias("fraud_rate"),
        col("month")
    )
    .orderBy("month")
)

fraud_trend = (
    transactions
    .groupBy(date_format(col("transaction_date"), "yyyy-MM").alias("month"))
    .agg(avg(when(col("is_fraud") == "Yes", 1).otherwise(0)).alias("fraud_rate"))
    .orderBy("month")
)

fraud_trend.write.mode("overwrite").saveAsTable("jarvis_etl.gold.fraud_rate_monthly_trend")

## Q3: Users with most flagged transactions

In [0]:
top_fraud_users = (
    transactions
    .filter(col("is_fraud") == "Yes")
    .groupBy("client_id")
    .count()
    .withColumnRenamed("count", "fraud_transaction_count")
    .orderBy(col("fraud_transaction_count").desc())
)

top_fraud_users.write.mode("overwrite").saveAsTable("jarvis_etl.gold.top_fraud_users")

## Q4: Users with sharp rise in transaction amount vs weekly average

In [0]:
from pyspark.sql.functions import weekofyear, year, avg, max

weekly_avg = (
    transactions
    .groupBy("client_id", year(col("transaction_date")).alias("yr"), weekofyear(col("transaction_date")).alias("week"))
    .agg(avg("amount").alias("weekly_avg_amount"), max("amount").alias("weekly_max_amount"))
)

spike_users = (
    weekly_avg
    .filter(col("weekly_max_amount") > col("weekly_avg_amount") * 3)
    .orderBy(col("weekly_max_amount").desc())
)

spike_users.write.mode("overwrite").saveAsTable("jarvis_etl.gold.users_amount_spikes")

## Q5: Merchant categories with highest fraud rate

In [0]:
fraud_by_mcc = (
    transactions
    .groupBy("mcc", "description")
    .agg(
        avg(when(col("is_fraud") == "Yes", 1).otherwise(0)).alias("fraud_rate"),
        col("mcc")
    )
)

fraud_by_mcc = (
    transactions
    .groupBy("mcc", "description")
    .agg(avg(when(col("is_fraud") == "Yes", 1).otherwise(0)).alias("fraud_rate"))
    .orderBy(col("fraud_rate").desc())
)

fraud_by_mcc.write.mode("overwrite").saveAsTable("jarvis_etl.gold.fraud_rate_by_merchant_category")

## Q6: Merchants with unusually high fraud volume

In [0]:
fraud_by_merchant = (
    transactions
    .filter(col("is_fraud") == "Yes")
    .groupBy("merchant_id", "merchant_city", "merchant_state")
    .count()
    .withColumnRenamed("count", "fraud_count")
    .orderBy(col("fraud_count").desc())
)

fraud_by_merchant.write.mode("overwrite").saveAsTable("jarvis_etl.gold.high_fraud_merchants")

## Q7: Fraud distribution by time of day

In [0]:
from pyspark.sql.functions import hour, when

fraud_by_hour = (
    transactions
    .filter(col("is_fraud") == "Yes")
    .groupBy(hour(col("transaction_date")).alias("hour_of_day"))
    .count()
    .withColumnRenamed("count", "fraud_count")
    .orderBy("hour_of_day")
)

fraud_by_hour.write.mode("overwrite").saveAsTable("jarvis_etl.gold.fraud_by_hour_of_day")

## Q8: Average transaction amount fraud vs non-fraud

In [0]:
from pyspark.sql.functions import avg

avg_amount = (
    transactions
    .groupBy("is_fraud")
    .agg(avg("amount").alias("avg_transaction_amount"))
)

avg_amount.write.mode("overwrite").saveAsTable("jarvis_etl.gold.avg_amount_fraud_vs_legit")

## Q9: Merchant category with highest total fraud amount"

In [0]:
from pyspark.sql.functions import sum

fraud_amount_by_mcc = (
    transactions
    .filter(col("is_fraud") == "Yes")
    .groupBy("mcc", "description")
    .agg(sum("amount").alias("total_fraud_amount"))
    .orderBy(col("total_fraud_amount").desc())
)

fraud_amount_by_mcc.write.mode("overwrite").saveAsTable("jarvis_etl.gold.total_fraud_amount_by_category")

## Q10: Total monetary losses due to fraud each day

In [0]:
from pyspark.sql.functions import to_date

daily_losses = (
    transactions
    .filter(col("is_fraud") == "Yes")
    .groupBy(to_date(col("transaction_date")).alias("transaction_date"))
    .agg(sum("amount").alias("total_fraud_amount"))
    .orderBy("transaction_date")
)

daily_losses.write.mode("overwrite").saveAsTable("jarvis_etl.gold.daily_fraud_losses")

## Q11: Unique users committing fraud per week

In [0]:
from pyspark.sql.functions import countDistinct

weekly_fraud_users = (
    transactions
    .filter(col("is_fraud") == "Yes")
    .groupBy(year(col("transaction_date")).alias("yr"), weekofyear(col("transaction_date")).alias("week"))
    .agg(countDistinct("client_id").alias("unique_fraud_users"))
    .orderBy("yr", "week")
)

weekly_fraud_users.write.mode("overwrite").saveAsTable("jarvis_etl.gold.weekly_unique_fraud_users")

## Q12: Monthly fraud spikes

In [0]:
monthly_fraud = (
    transactions
    .filter(col("is_fraud") == "Yes")
    .groupBy(date_format(col("transaction_date"), "yyyy-MM").alias("month"))
    .count()
    .withColumnRenamed("count", "fraud_count")
    .orderBy("month")
)

monthly_fraud.write.mode("overwrite").saveAsTable("jarvis_etl.gold.monthly_fraud_spikes")

## Q13: User behavior before vs after fraudulent event

In [0]:
from pyspark.sql.functions import min, lag
from pyspark.sql.window import Window

first_fraud = (
    transactions
    .filter(col("is_fraud") == "Yes")
    .groupBy("client_id")
    .agg(min("transaction_date").alias("first_fraud_date"))
)

user_behavior = (
    transactions
    .join(first_fraud, "client_id")
    .withColumn("period", when(col("transaction_date") < col("first_fraud_date"), "before_fraud").otherwise("after_fraud"))
    .groupBy("client_id", "period")
    .agg(avg("amount").alias("avg_amount"), col("period"))
)

user_behavior = (
    transactions
    .join(first_fraud, "client_id")
    .withColumn("period", when(col("transaction_date") < col("first_fraud_date"), "before_fraud").otherwise("after_fraud"))
    .groupBy("client_id", "period")
    .agg(avg("amount").alias("avg_amount"))
)

user_behavior.write.mode("overwrite").saveAsTable("jarvis_etl.gold.user_behavior_before_after_fraud")

## Q14: Fraud more common on high value purchases

In [0]:
from pyspark.sql.functions import percentile_approx

threshold = transactions.agg(percentile_approx("amount", 0.75)).collect()[0][0]

fraud_by_value = (
    transactions
    .withColumn("purchase_tier", when(col("amount") >= threshold, "high_value").otherwise("normal_value"))
    .groupBy("purchase_tier")
    .agg(
        avg(when(col("is_fraud") == "Yes", 1).otherwise(0)).alias("fraud_rate"),
        col("purchase_tier")
    )
)

fraud_by_value = (
    transactions
    .withColumn("purchase_tier", when(col("amount") >= threshold, "high_value").otherwise("normal_value"))
    .groupBy("purchase_tier")
    .agg(avg(when(col("is_fraud") == "Yes", 1).otherwise(0)).alias("fraud_rate"))
)

fraud_by_value.write.mode("overwrite").saveAsTable("jarvis_etl.gold.fraud_rate_by_purchase_tier")

In [0]:
%sql
SHOW TABLES IN jarvis_etl.gold;

database,tableName,isTemporary
gold,avg_amount_fraud_vs_legit,false
gold,daily_fraud_losses,false
gold,fraud_by_day_of_week,false
gold,fraud_by_hour_of_day,false
gold,fraud_rate_by_merchant_category,false
gold,fraud_rate_by_purchase_tier,false
gold,fraud_rate_monthly_trend,false
gold,high_fraud_merchants,false
gold,monthly_fraud_spikes,false
gold,top_fraud_users,false
